# Bank Marketing Campaign Analysis — DA-014

This notebook performs a reproducible exploratory analysis of UCI's `bank-additional-full.csv`. It uses the same `analysis.py` functions as the Streamlit dashboard, so data cleaning, response-rate denominators, and segment calculations are consistent.

**Target:** `y` (`yes` = subscribed to a term deposit; `no` = did not subscribe). **Rate denominator:** all records with a valid yes/no target in the current cohort. Each row is a contact record, not necessarily a unique person. Source and attribution: [`data/SOURCE.md`](data/SOURCE.md).

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px

# Run Jupyter from the project folder so the shared analysis module and CSV are found.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'analysis.py').exists():
    raise FileNotFoundError('Start Jupyter from the extracted DA-014 project folder so analysis.py and data/ are available.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from analysis import (
    CATEGORICAL_COLUMNS, apply_filters, calculate_kpis,
    contact_frequency_table, generate_findings, load_data, quality_table,
    response_by_segment,
)

data, quality = load_data()
print(f'Loaded {len(data):,} valid contact records and {data.shape[1]} columns.')

Loaded 41,188 valid contact records and 21 columns.


## 1. Cleaning and data quality

The cleaning rules preserve `Unknown` categories, convert `?` to missing, parse numeric fields, and remove only rows with an invalid target.

In [2]:
quality_table(quality)

,Data-quality measure,Value
0,Rows read from CSV,41188
1,Rows retained with valid target,41188
2,Rows excluded for invalid target,0
3,Missing cells after cleaning,0
4,Explicit Unknown categories,12718


In [3]:
display(data.head())
display(data.dtypes.rename('dtype').to_frame())
display(data['y'].value_counts(dropna=False).rename_axis('outcome').to_frame('records'))

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,Unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


,dtype
age,int64
job,string
marital,string
education,string
default,string
housing,string
loan,string
contact,string
month,string
day_of_week,string


,records
outcome,
no,36548
yes,4640


## 2. Overall campaign response rate

The denominator includes all retained records with a valid target. The rate is computed directly from `y`; no target balance or KPI is hard-coded.

In [4]:
overall = calculate_kpis(data)
pd.DataFrame([
    {'Metric': 'Records analyzed', 'Value': f"{overall['records']:,}"},
    {'Metric': 'Successful responses (y = yes)', 'Value': f"{overall['responses']:,}"},
    {'Metric': 'Response rate', 'Value': f"{overall['response_rate']:.2%}"},
    {'Metric': 'Average contacts during this campaign', 'Value': f"{overall['average_campaign_contacts']:.2f}"},
])

,Metric,Value
0,Records analyzed,"41,188"
1,Successful responses (y = yes),"4,640"
2,Response rate,11.27%
3,Average contacts during this campaign,2.57


## 3. Response rates across customer and campaign segments

Each result includes response counts and record denominators so percentages can be checked. Small groups should be interpreted cautiously.

In [5]:
segment_tables = {}
for column in CATEGORICAL_COLUMNS:
    table = response_by_segment(data, column)
    segment_tables[column] = table
    print(f'\nResponse by {column}:')
    display(table.style.format({'response_rate': '{:.2%}'}))


Response by job:

,segment,records,responses,response_rate
0,student,875,275,31.43%
1,retired,1720,434,25.23%
2,unemployed,1014,144,14.20%
3,admin.,10422,1352,12.97%
4,management,2924,328,11.22%
5,Unknown,330,37,11.21%
6,technician,6743,730,10.83%
7,self-employed,1421,149,10.49%
8,housemaid,1060,106,10.00%
9,entrepreneur,1456,124,8.52%



Response by marital:


,segment,records,responses,response_rate
0,Unknown,80,12,15.00%
1,single,11568,1620,14.00%
2,divorced,4612,476,10.32%
3,married,24928,2532,10.16%



Response by education:



Response by day_of_week:


,segment,records,responses,response_rate
0,thu,8623,1045,12.12%
1,tue,8090,953,11.78%
2,wed,8134,949,11.67%
3,fri,7827,846,10.81%
4,mon,8514,847,9.95%



Response by poutcome:


,segment,records,responses,response_rate
0,success,1373,894,65.11%
1,failure,4252,605,14.23%
2,nonexistent,35563,3141,8.83%


In [6]:
job_rates = response_by_segment(data, 'job').sort_values('response_rate')
fig = px.bar(job_rates, x='response_rate', y='segment', orientation='h', hover_data=['records', 'responses'], title='Term-deposit response rate by job')
fig.update_xaxes(tickformat='.0%', title='Response rate')
fig.show()

## 4. Contact frequency, channel, and timing

`campaign` is the number of contacts during this campaign, including the last contact. Channel, month, and weekday comparisons are descriptive and can be affected by customer mix and timing.

In [7]:
frequency = contact_frequency_table(data)
display(frequency.style.format({'response_rate': '{:.2%}'}))
fig = px.bar(frequency, x='contact_band', y='response_rate', hover_data=['records', 'responses'], title='Response rate by number of campaign contacts')
fig.update_yaxes(tickformat='.0%', title='Response rate')
fig.show()

for column in ['contact', 'month', 'day_of_week']:
    table = response_by_segment(data, column)
    label = column.replace('_', ' ')
    print(f'\n{label.title()} comparison:')
    display(table.style.format({'response_rate': '{:.2%}'}))
    fig = px.bar(table, x='segment', y='response_rate', hover_data=['records', 'responses'], title=f'Response rate by {label}')
    fig.update_yaxes(tickformat='.0%', title='Response rate')
    fig.show()

,contact_band,records,responses,response_rate
0,1,17642,2300,13.04%
1,2,10570,1211,11.46%
2,3,5341,574,10.75%
3,4,2651,249,9.39%
4,5+,4984,306,6.14%



Contact comparison:


,segment,records,responses,response_rate
0,cellular,26144,3853,14.74%
1,telephone,15044,787,5.23%



Month comparison:


,segment,records,responses,response_rate
0,mar,546,276,50.55%
1,dec,182,89,48.90%
2,sep,570,256,44.91%
3,oct,718,315,43.87%
4,apr,2632,539,20.48%
5,aug,6178,655,10.60%
6,jun,5318,559,10.51%
7,nov,4101,416,10.14%
8,jul,7174,649,9.05%
9,may,13769,886,6.43%



Day Of Week comparison:


,segment,records,responses,response_rate
0,thu,8623,1045,12.12%
1,tue,8090,953,11.78%
2,wed,8134,949,11.67%
3,fri,7827,846,10.81%
4,mon,8514,847,9.95%


## 5. Interactive cohort example

The dashboard supports all available category filters and numeric ranges. This cell demonstrates the same shared filter function with an example cohort.

In [8]:
# Example: cellular contacts, ages 30–60. Edit these values to explore another cohort.
example = apply_filters(data, {'contact': ['cellular']}, {'age': (30, 60)})
example_kpis = calculate_kpis(example)
print(f"Records: {example_kpis['records']:,}; successful responses: {example_kpis['responses']:,}; response rate: {example_kpis['response_rate']:.2%}")
display(example.head(10))

Records: 21,508; successful responses: 2,670; response rate: 12.41%

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
12357,45,blue-collar,married,basic.9y,no,yes,no,cellular,jul,fri,...,1,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12360,33,technician,married,basic.9y,Unknown,yes,no,cellular,jul,fri,...,4,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12361,41,management,married,university.degree,no,yes,no,cellular,jul,fri,...,3,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12363,30,services,divorced,high.school,no,yes,yes,cellular,jul,fri,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12364,58,retired,married,professional.course,Unknown,yes,no,cellular,jul,fri,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12365,31,blue-collar,married,basic.9y,no,Unknown,Unknown,cellular,jul,fri,...,3,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12366,30,admin.,married,high.school,no,yes,no,cellular,jul,fri,...,6,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12367,40,blue-collar,divorced,basic.6y,Unknown,Unknown,Unknown,cellular,jul,fri,...,3,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12368,33,services,married,high.school,no,yes,no,cellular,jul,fri,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
12369,38,blue-collar,married,basic.9y,no,no,no,cellular,jul,fri,...,2,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no


## 6. Findings and recommendations

Findings require at least 30 records per segment. Recommendations remain hypotheses for testing; observed rates do not establish a causal effect.

In [9]:
for item in generate_findings(data):
    print(f"\n{item['title']}")
    print(f"Finding: {item['finding']}")
    print(f"Suggested next step: {item['recommendation']}")


Highest observed response: Job


Finding: student has the highest observed rate in the selected cohort: 275 of 875 records (31.4%). Groups below 30 records are omitted.
Suggested next step: Use this segment as a hypothesis for a targeted campaign test; this descriptive association does not establish that the segment caused the response.

Contact-frequency comparison
Finding: Response was 13.0% after one contact (17,642 records) and 6.1% after 5+ contacts (4,984 records); the 5+ band was 6.9 percentage points lower.
Suggested next step: Review the contact cadence for the one contact group before changing follow-up rules; the comparison is observational and may reflect differences in who was contacted.

Communication-channel comparison
Finding: cellular has the highest observed channel rate at 14.7% (3,853/26,144); the comparison channel telephone is 5.2% (787/15,044).
Suggested next step: Use channel differences to design a controlled follow-up test; customer mix and campaign timing can also explain the gap.


## 7. Limitations and interpretation

- `duration` is call duration in seconds and is only known after contact; it should not inform pre-call targeting.
- `pdays` uses the source dataset's encoding; 999 indicates the client was not previously contacted.
- A row is a contact record, not a verified unique customer.
- This is historical observational data. Patterns may reflect selection, customer mix, timing, and other confounders. Validate proposed campaign changes with controlled experiments.
- Source: Moro, S., Rita, P., & Cortez, P. (2014). Bank Marketing. UCI Machine Learning Repository. DOI: [10.24432/C5K306](https://doi.org/10.24432/C5K306). License: CC BY 4.0.